# Part 3: Retrieval Models

**CS533: Information Retrieval Systems**

This notebook examines retrieval methods on the two NYT test collections:

- **C1:** 2005–2014
- **C2:** 2015–2025

We examine lexical retrieval (TF-IDF and BM25), dense retrieval using word embeddings, query expansion, hybrid retrieval, and efficiency.

All experiments use the saved retrieval results. This notebook verifies the methods and presents the required findings without rebuilding indexes or rerunning retrieval experiments.

In [1]:

from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
from IPython.display import display

# Project and saved experimental artifacts.
ROOT = Path(r"C:\Projects\IR\CS533_HW1")
DATA = ROOT.parent / "data"

OFFICIAL = DATA / "nyt-tc-2026"
RESULTS = DATA / "part3_results"
INDEXES = DATA / "part3_indexes"

# Make the project's preprocessing code importable.
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.preprocessing import preprocess_text

# Check paths without loading or rebuilding indexes.
paths = {
    "Project": ROOT,
    "Official test collection": OFFICIAL,
    "Retrieval results": RESULTS,
    "Saved indexes": INDEXES,
}

path_check = pd.DataFrame([
    {"Artifact": name, "Exists": path.is_dir()}
    for name, path in paths.items()
])

display(path_check)
print("All directories found:", path_check["Exists"].all())


,Artifact,Exists
0,Project,True
1,Official test collection,True
2,Retrieval results,True
3,Saved indexes,True


All directories found: True


### 3.1 Lexical Baselines

We use two lexical retrieval methods:

- **TF-IDF:** Cosine similarity with sublinear term frequency and the shared `preprocess_text` analyzer.
- **BM25:** Term-frequency saturation and document-length normalization with \(k_1=1.2\) and \(b=0.75\).

Separate indexes are used for C1 and C2. Each collection is searched using its corresponding known-item queries and the same 32 topic queries.

This section verifies the saved top-100 retrieval results without rebuilding the indexes.

In [2]:

# Load official collection and query metadata.
collections = {
    "C1": pd.read_csv(OFFICIAL / "collection_P1.tsv", sep="\t"),
    "C2": pd.read_csv(OFFICIAL / "collection_P2.tsv", sep="\t")
}

known_items = {
    "C1": pd.read_csv(OFFICIAL / "queries_ki_P1.tsv", sep="\t"),
    "C2": pd.read_csv(OFFICIAL / "queries_ki_P2.tsv", sep="\t")
}

topics = pd.read_csv(OFFICIAL / "topics.tsv", sep="\t")

# Expected document and query IDs for each collection.
collection_ids = {
    name: set(df["docno"])
    for name, df in collections.items()
}

expected_queries = {
    name: set(known_items[name]["qid"]) | set(topics["tid"])
    for name in collections
}

# Display collection information.
collection_summary = pd.DataFrame([
    {
        "Collection": name,
        "Documents": len(collections[name]),
        "Known-item queries": len(known_items[name]),
        "Topic queries": len(topics)
    }
    for name in ("C1", "C2")
])

print("Collection Summary")
display(collection_summary)


# Validate a previously saved top-100 retrieval run.
def audit_run(filename, collection):
    path = RESULTS / filename

    if not path.is_file():
        return {
            "File": filename,
            "Collection": collection,
            "Passed": False,
            "Issue": "File missing"
        }

    run = pd.read_csv(path)
    required = ["qid", "docno", "rank", "score"]

    if not set(required).issubset(run.columns):
        return {
            "File": filename,
            "Collection": collection,
            "Passed": False,
            "Issue": "Missing required columns"
        }

    by_query = run.groupby("qid")
    ordered = run.sort_values(["qid", "rank"])

    checks = {
        "Row count": len(run) == 103_200,
        "Query IDs": set(run["qid"]) == expected_queries[collection],
        "Document IDs": run["docno"].isin(
            collection_ids[collection]
        ).all(),
        "100 per query": by_query.size().eq(100).all(),
        "Unique documents": by_query["docno"].nunique().eq(100).all(),
        "Valid ranks": (
            by_query["rank"].min().eq(1).all()
            and by_query["rank"].max().eq(100).all()
            and by_query["rank"].nunique().eq(100).all()
            and run["rank"].mod(1).eq(0).all()
        ),
        "No missing values": run[required].notna().all().all(),
        "Finite scores": np.isfinite(run["score"]).all(),
        "Descending scores": (
            ordered.groupby("qid")["score"]
            .diff().dropna().le(1e-7).all()
        )
    }

    failures = [key for key, passed in checks.items() if not passed]

    return {
        "File": filename,
        "Collection": collection,
        "Passed": not failures,
        "Issue": ", ".join(failures)
    }


# Verify both lexical methods on both collections.
lexical_results = [
    audit_run(f"{coll.lower()}_{method}.csv", coll)
    for coll in ("C1", "C2")
    for method in ("tfidf", "bm25")
]

lexical_audit = pd.DataFrame(lexical_results)

print("\nLexical Retrieval Validation")
display(lexical_audit)

print(
    f"\nPassed: {lexical_audit['Passed'].sum()}"
    f"/{len(lexical_audit)}"
)


Collection Summary


,Collection,Documents,Known-item queries,Topic queries
0,C1,173603,1000,32
1,C2,210664,1000,32



Lexical Retrieval Validation


,File,Collection,Passed,Issue
0,c1_tfidf.csv,C1,True,
1,c1_bm25.csv,C1,True,
2,c2_tfidf.csv,C2,True,
3,c2_bm25.csv,C2,True,



Passed: 4/4


#### 3.1 Key Findings

TF-IDF and BM25 retrieval were implemented separately for C1 (173,603 documents) and C2 (210,664 documents).

TF-IDF uses sublinear term frequency and cosine similarity, while BM25 uses \(k_1=1.2\) and \(b=0.75\). Both use the shared preprocessing pipeline.

All four saved retrieval runs passed validation, returning **100 documents for each of 1,032 queries per collection**, with valid ranks, document IDs, and scores.

### 3.2 Dense Retrieval with Averaged Embeddings

Dense retrieval represents documents and queries using TF-IDF-weighted averages of word embeddings, followed by L2 normalization and cosine similarity.

We compare M1, M2, M2s, and GN, reporting out-of-vocabulary (OOV) token fractions, queries containing OOV terms, and queries with no usable embedding.

In [5]:

# Locate saved dense retrieval summaries.
summary_files = sorted(RESULTS.rglob("dense_summary.json"))

print(f"Dense retrieval summaries found: {len(summary_files)}")

for path in summary_files:
    print(path.relative_to(RESULTS))

# Inspect one report to identify its OOV statistics.
if summary_files:
    sample_path = summary_files[0]
    sample = json.loads(sample_path.read_text(encoding="utf-8"))

    print(f"\nExample report: {sample_path.parent.name}")
    print("\nAvailable fields:")

    for key, value in sample.items():
        if isinstance(value, dict):
            print(f"{key}: {list(value.keys())}")
        elif isinstance(value, list):
            print(f"{key}: list with {len(value)} entries")
        else:
            print(f"{key}: {value}")
else:
    print("No dense_summary.json files found.")


Dense retrieval summaries found: 8
dense_c1_gn\dense_summary.json
dense_c1_m1\dense_summary.json
dense_c1_m2\dense_summary.json
dense_c1_m2s\dense_summary.json
dense_c2_gn\dense_summary.json
dense_c2_m1\dense_summary.json
dense_c2_m2\dense_summary.json
dense_c2_m2s\dense_summary.json

Example report: dense_c1_gn

Available fields:
model: GN
collection: C1
smoke: False
document_cache: C:\Projects\IR\data\part3_indexes\C1_GN
cache_reused: False
model_loading_seconds: 0.25873069999943255
document_embeddings: ['provenance', 'documents', 'idf_document_count', 'google_news', 'embedding_lookup', 'idf_convention', 'term_frequency', 'normalized', 'embedding_seconds', 'document_matrix_bytes', 'documents_sha256', 'docnos_sha256', 'memory_after_embedding', 'matched_documents', 'missing_article_texts', 'joined_docno_text_sha256', 'zero_document_embeddings', 'observed_terms_with_embeddings', 'document_matrix_shape', 'document_matrix_dtype']
runs: list with 2 entries
memory_before: ['rss_bytes', 'pro

In [6]:

# Inspect the saved query-level statistics.
sample_path = RESULTS / "dense_c1_gn" / "dense_summary.json"

sample = json.loads(
    sample_path.read_text(encoding="utf-8")
)

print("Model:", sample["model"])
print("Collection:", sample["collection"])
print("Number of runs:", len(sample["runs"]))

for i, run in enumerate(sample["runs"], start=1):
    print(f"\n--- Run {i} ---")

    for key, value in run.items():
        if isinstance(value, dict):
            print(f"{key}: {list(value.keys())}")

            # Inspect one additional level of nested statistics.
            for subkey, subvalue in value.items():
                if isinstance(subvalue, dict):
                    print(
                        f"  {key}.{subkey}: "
                        f"{list(subvalue.keys())}"
                    )

        elif isinstance(value, list):
            print(f"{key}: list with {len(value)} entries")

        else:
            print(f"{key}: {value}")


Model: GN
Collection: C1
Number of runs: 2

--- Run 1 ---
model: GN
collection: C1
query_set: known_item
filename: c1_gn_known_item.csv
queries: 1000
rows: 100000
verification: ['passed', 'queries', 'rows', 'unique_official_documents_per_query']
oov: ['queries', 'query_tokens', 'oov_tokens', 'queries_with_oov', 'queries_without_embedding', 'oov_token_fraction', 'queries_with_oov_fraction']
query_seconds: 4.825262600003043
mean_query_seconds: 0.004825262600003043
query_timing: Embedding, batched cosine scoring, ranking; excludes CSV writing/validation
maximum_score_matrix_shape: list with 2 entries
maximum_score_matrix_bytes: 11110592
memory_after_queries: ['rss_bytes', 'process_peak_rss_bytes', 'system_available_bytes', 'system_total_bytes']

--- Run 2 ---
model: GN
collection: C1
query_set: topics
filename: c1_gn_topics.csv
queries: 32
rows: 3200
verification: ['passed', 'queries', 'rows', 'unique_official_documents_per_query']
oov: ['queries', 'query_tokens', 'oov_tokens', 'queries_w

In [8]:

# Report saved OOV statistics for all four embedding models.
# No models or retrieval indexes are rebuilt.

oov_rows = []

for path in sorted(summary_files):
    report = json.loads(path.read_text(encoding="utf-8"))

    collection = report["collection"]
    model = report["model"]

    for run in report["runs"]:
        stats = run["oov"]

        required = [
            "queries",
            "query_tokens",
            "oov_tokens",
            "queries_with_oov",
            "queries_without_embedding",
            "oov_token_fraction",
            "queries_with_oov_fraction"
        ]

        assert all(key in stats for key in required)
        assert stats["queries"] == run["queries"]
        assert stats["query_tokens"] > 0

        oov_rows.append({
            "Collection": collection,
            "Query set": run["query_set"],
            "Model": model,
            "OOV tokens (%)": round(
                stats["oov_token_fraction"] * 100, 2
            ),
            "Queries with OOV (%)": round(
                stats["queries_with_oov_fraction"] * 100, 2
            ),
            "No embedding": stats["queries_without_embedding"]
        })

# Organize the results for reporting.
oov_table = pd.DataFrame(oov_rows)

model_order = ["M1", "M2", "M2s", "GN"]
query_order = ["known_item", "topics"]

oov_table["Model"] = pd.Categorical(
    oov_table["Model"],
    categories=model_order,
    ordered=True
)

oov_table["Query set"] = pd.Categorical(
    oov_table["Query set"],
    categories=query_order,
    ordered=True
)

oov_table = (
    oov_table
    .sort_values(["Collection", "Query set", "Model"])
    .reset_index(drop=True)
)

print("Dense Retrieval: Query OOV Statistics")
display(oov_table)

# Verify every expected combination is present.
expected = {
    (collection, query_set, model)
    for collection in ["C1", "C2"]
    for query_set in query_order
    for model in model_order
}

actual = set(zip(
    oov_table["Collection"].astype(str),
    oov_table["Query set"].astype(str),
    oov_table["Model"].astype(str)
))

assert actual == expected
assert len(oov_table) == 16
assert oov_table["No embedding"].ge(0).all()
assert oov_table["OOV tokens (%)"].between(0, 100).all()
assert oov_table["Queries with OOV (%)"].between(0, 100).all()

print("\nAll 16 OOV reporting combinations verified.")


Dense Retrieval: Query OOV Statistics


,Collection,Query set,Model,OOV tokens (%),Queries with OOV (%),No embedding
0,C1,known_item,M1,3.41,22.7,0
1,C1,known_item,M2,3.36,22.3,0
2,C1,known_item,M2s,4.74,29.5,0
3,C1,known_item,GN,0.99,7.1,0
4,C1,topics,M1,0.00,0.0,0
5,C1,topics,M2,0.00,0.0,0
6,C1,topics,M2s,0.00,0.0,0
7,C1,topics,GN,0.00,0.0,0
8,C2,known_item,M1,5.87,25.8,0
9,C2,known_item,M2,3.52,15.4,0



All 16 OOV reporting combinations verified.


#### 3.2 Key Findings

- **M2:** Continued training improves C2 vocabulary coverage, reducing the known-item OOV token rate from 5.87% (M1) to 3.52%.
- **Google News:** Achieves the lowest OOV rates for known-item queries on both collections (0.99% for C1 and 1.56% for C2).
- **Topic queries:** All four models have 0% OOV tokens across both collections.
- **Missing embeddings:** No queries lack usable embeddings in either collection.

**Conclusion:** M2 improves vocabulary coverage for P2 queries, while GN provides the broadest coverage. However, lower OOV rates do not necessarily indicate better retrieval performance.

### 3.3 Embedding-Based Query Expansion

Query expansion (QE) adds semantically related terms using the centroid of normalized query word embeddings. Candidate terms are restricted to the collection vocabulary, excluding original query terms.

We use M1, M2, and GN with \(m \in \{5,10,20\}\) expansion terms and \(\lambda \in \{0.5,1.0\}\). Original query terms receive weight 1, while expansion terms are weighted by their cosine similarity multiplied by \(\lambda\).

This section reports expansion terms for five topic queries using each model on C1 and C2.

In [9]:

# Load saved topic query expansion results.
qe_path = RESULTS / "qe_topic_expansions.json"
assert qe_path.is_file(), "Saved QE results not found."

qe_data = json.loads(qe_path.read_text(encoding="utf-8"))

models = ["M1", "M2", "GN"]
collections_to_check = ["C1", "C2"]

for collection in collections_to_check:
    rows = {}

    for model in models:
        records = qe_data["collections"][collection][model]["topics"]

        assert len(records) == 5

        for record in records:
            qid = record["qid"]
            query = record["query"]
            neighbors = record["neighbors"]
            original_tokens = set(record["tokens"])

            assert len(neighbors) >= 5

            terms = [item["term"] for item in neighbors]
            scores = [float(item["cosine"]) for item in neighbors]

            # Verify expansion terms.
            assert len(terms) == len(set(terms))
            assert not original_tokens.intersection(terms)
            assert np.isfinite(scores).all()
            assert all(
                scores[i] >= scores[i + 1] - 1e-7
                for i in range(len(scores) - 1)
            )

            key = (qid, query)
            rows.setdefault(key, {})[model] = ", ".join(terms[:5])

    assert len(rows) == 5
    assert all(set(values) == set(models) for values in rows.values())

    expansion_table = pd.DataFrame([
        {
            "Topic ID": qid,
            "Query": query,
            **values
        }
        for (qid, query), values in sorted(rows.items())
    ])

    print(f"\n{collection}: Top 5 Expansion Terms")

    with pd.option_context("display.max_colwidth", None):
        display(expansion_table)

print("\nAll topic expansion checks passed.")



C1: Top 5 Expansion Terms


,Topic ID,Query,M1,M2,GN
0,T01,africa,"saharan, african, bloemfontein, africans, lesotho","african, sahel, pretoria, bloemfontein, saharan","african, zimbabwe, america, kenya, africans"
1,T02,art,"contemporary, museum, heckscher, galleries, gallery","museum, galleries, contemporary, exhibitions, gallery","artworks, artwork, printmaking, arts, artmaking"
2,T04,baseball,"league, selig, chass, leagues, sluggers","mlb, league, solarte, leagues, pitcher","softball, ballplayers, basketball, football, ballplayer"
3,T23,politics,"lawmaking, political, helpline, donnybrook, rightward","lerer, tribalism, crosscurrents, centrism, helpline","political, partisanship, politicians, politician, politicking"
4,T27,technology,"technologies, coders, tech, virtualization, computing","tech, technologies, coders, generative, software","technologies, innovations, innovation, advancements, techology"



C2: Top 5 Expansion Terms


,Topic ID,Query,M1,M2,GN
0,T01,africa,"saharan, african, africans, lesotho, uganda","african, sahel, pretoria, saharan, johannesburg","african, zimbabwe, america, kenya, africans"
1,T02,art,"contemporary, museum, heckscher, galleries, gallery","museum, galleries, contemporary, exhibitions, gallery","artworks, artwork, printmaking, arts, artmaking"
2,T04,baseball,"league, selig, leagues, sluggers, yankees","mlb, league, solarte, leagues, pitcher","softball, ballplayers, basketball, football, ballplayer"
3,T23,politics,"lawmaking, political, rightward, polarization, politicians","lerer, corasaniti, tribalism, crosscurrents, centrism","political, partisanship, politicians, politician, politicking"
4,T27,technology,"technologies, coders, tech, computing, software","tech, technologies, coders, generative, software","technologies, innovations, innovation, advancements, tech"



All topic expansion checks passed.


In [10]:

# Check whether the original query appears among GN expansions.
for collection in ["C1", "C2"]:
    records = qe_data["collections"][collection]["GN"]["topics"]

    record = next(
        r for r in records if r["query"] == "technology"
    )

    query_tokens = set(preprocess_text(record["query"]))
    expansion_terms = [
        item["term"] for item in record["neighbors"][:5]
    ]

    print(f"\n{collection} GN: technology")
    print("Recorded tokens:", record["tokens"])
    print("Actual query tokens:", query_tokens)
    print("Expansion terms:", expansion_terms)
    print(
        "Original terms included:",
        query_tokens.intersection(expansion_terms)
    )



C1 GN: technology
Recorded tokens: ['technology']
Actual query tokens: {'technology'}
Expansion terms: ['technologies', 'innovations', 'innovation', 'advancements', 'techology']
Original terms included: set()

C2 GN: technology
Recorded tokens: ['technology']
Actual query tokens: {'technology'}
Expansion terms: ['technologies', 'innovations', 'innovation', 'advancements', 'tech']
Original terms included: set()


#### 3.3 Key Findings

- **M1:** Produces expansions reflecting earlier contexts, such as `league` and `selig` for baseball and `coders` and `virtualization` for technology.
- **M2:** Incorporates newer associations, including `mlb` for baseball and `generative` for technology.
- **Google News:** Produces broader expansions, such as `artworks` for art and `innovation` for technology.
- **Limitations:** Some expansions are noisy or less relevant, particularly `techology` (GN) and `lerer` (M2). Semantic similarity does not guarantee useful retrieval terms.

**Conclusion:** Expansion terms differ across embedding models and periods. Whether these differences improve retrieval effectiveness will be evaluated in Part 4.

### 3.4 Hybrid Retrieval

Hybrid retrieval combines BM25 and dense retrieval using the union of their top 100 documents as candidates.

We evaluate two fusion methods:
- **Linear fusion:** Combines min-max normalized BM25 and dense scores using α from 0 to 1 in increments of 0.1.
- **Reciprocal Rank Fusion (RRF):** Combines document rankings using a constant of 60, assigning no contribution for documents absent from a ranker's top 100.

We use M1, M2, and GN with BM25 on both collections. This section verifies the saved hybrid retrieval outputs.

In [11]:

# Verify saved hybrid retrieval runs without recomputing scores.
hybrid_rows = []

for collection in ["C1", "C2"]:
    for model in ["m1", "m2", "gn"]:

        # Linear fusion: alpha = 0.0, 0.1, ..., 1.0
        linear_files = [
            RESULTS / (
                f"{collection.lower()}_hybrid_{model}_"
                f"a{tenth // 10}p{tenth % 10}.csv"
            )
            for tenth in range(11)
        ]

        # Reciprocal Rank Fusion.
        rrf_file = (
            RESULTS / f"{collection.lower()}_rrf_{model}.csv"
        )

        hybrid_rows.append({
            "Collection": collection,
            "Model": model.upper(),
            "Linear runs": sum(
                path.is_file() and path.stat().st_size > 0
                for path in linear_files
            ),
            "RRF runs": int(
                rrf_file.is_file()
                and rrf_file.stat().st_size > 0
            ) if rrf_file.is_file() else 0
        })

hybrid_summary = pd.DataFrame(hybrid_rows)

print("Hybrid Retrieval: Saved Run Summary")
display(hybrid_summary)

print(f"\nLinear runs: {hybrid_summary['Linear runs'].sum()}/66")
print(f"RRF runs: {hybrid_summary['RRF runs'].sum()}/6")

assert hybrid_summary["Linear runs"].eq(11).all()
assert hybrid_summary["RRF runs"].eq(1).all()

print("\nAll 72 expected hybrid files exist and are nonempty.")


Hybrid Retrieval: Saved Run Summary


,Collection,Model,Linear runs,RRF runs
0,C1,M1,11,1
1,C1,M2,11,1
2,C1,GN,11,1
3,C2,M1,11,1
4,C2,M2,11,1
5,C2,GN,11,1



Linear runs: 66/66
RRF runs: 6/6

All 72 expected hybrid files exist and are nonempty.


#### 3.4 Key Findings

All **66 linear-fusion runs** and **6 RRF runs** are present for C1 and C2 using M1, M2, and GN.

**Candidate selection:** The union of BM25's top 100 and dense retrieval's top 100 allows both methods to contribute documents. Restricting candidates to BM25 alone would exclude potentially relevant documents identified only through semantic similarity.

**Fusion methods:** Linear fusion combines min-max normalized scores using α values from 0 to 1. RRF combines rankings using a constant of 60, with no contribution from a ranker when a document is absent from its top 100.

The effect of these fusion strategies on retrieval effectiveness is evaluated in Part 4.

### 3.5 Efficiency

We compare indexing time, dense document matrix memory, and average query latency for BM25, dense retrieval, query expansion, and hybrid retrieval.

Measurements come from previously completed experiments. We distinguish preprocessing and indexing costs, query execution time, and fusion-only overhead to ensure fair comparisons.

In [12]:

# Read recorded dense retrieval efficiency measurements.
# No retrieval or indexing operations are performed.

efficiency_rows = []

for path in sorted(summary_files):
    report = json.loads(path.read_text(encoding="utf-8"))

    collection = report["collection"]
    model = report["model"]
    document_info = report["document_embeddings"]

    # Dense document matrix storage.
    matrix_bytes = document_info["document_matrix_bytes"]
    matrix_mib = matrix_bytes / (1024 ** 2)

    # Document embedding construction time.
    embedding_seconds = document_info["embedding_seconds"]

    # Separate known-item and topic query timing.
    for run in report["runs"]:
        efficiency_rows.append({
            "Collection": collection,
            "Model": model,
            "Query set": run["query_set"],
            "Embedding construction (s)": embedding_seconds,
            "Dense matrix (MiB)": round(matrix_mib, 2),
            "Average query (ms)": round(
                run["mean_query_seconds"] * 1000, 3
            )
        })

dense_efficiency = pd.DataFrame(efficiency_rows)

print("Dense Retrieval Efficiency")
display(
    dense_efficiency.sort_values(
        ["Collection", "Model", "Query set"]
    ).reset_index(drop=True)
)

assert len(dense_efficiency) == 16
assert dense_efficiency["Dense matrix (MiB)"].gt(0).all()

print("\nAll 16 dense efficiency records verified.")

# Inspect available timing sources for the remaining methods.
print("\nHybrid timing metadata available:")
print(
    (RESULTS / "hybrid_clarification_validation.json").is_file()
)

print("\nIndex metadata files:")
metadata_files = [
    path for path in INDEXES.rglob("*.json")
    if any(term in path.name.lower()
           for term in ("summary", "metadata", "timing"))
]

for path in sorted(metadata_files)[:15]:
    print(path.relative_to(INDEXES))

print(f"Matching index metadata files: {len(metadata_files)}")


Dense Retrieval Efficiency


,Collection,Model,Query set,Embedding construction (s),Dense matrix (MiB),Average query (ms)
0,C1,GN,known_item,77.131066,198.67,4.825
1,C1,GN,topics,77.131066,198.67,4.512
2,C1,M1,known_item,69.119673,198.67,4.627
3,C1,M1,topics,69.119673,198.67,3.958
4,C1,M2,known_item,70.056397,198.67,4.611
5,C1,M2,topics,70.056397,198.67,4.436
6,C1,M2s,known_item,75.872504,198.67,4.954
7,C1,M2s,topics,75.872504,198.67,5.051
8,C2,GN,known_item,77.885656,241.09,5.676
9,C2,GN,topics,77.885656,241.09,5.890



All 16 dense efficiency records verified.

Hybrid timing metadata available:
True

Index metadata files:
Matching index metadata files: 0


In [17]:

# Section 3.5: QE, linear fusion, and RRF efficiency.
# Read saved measurements without rerunning experiments.

qe_data = json.loads(
    (RESULTS / "qe_summary.json").read_text(encoding="utf-8")
)

hybrid_data = json.loads(
    (RESULTS / "hybrid_clarification_validation.json")
    .read_text(encoding="utf-8")
)

hybrid_full = json.loads(
    (RESULTS / "hybrid_summary.json").read_text(encoding="utf-8")
)

# 1. Query expansion timing.
qe_rows = []

for record in qe_data["results"]:
    qe_rows.append({
        "Collection": record["collection"],
        "Model": record["model"].upper(),
        "QE query (ms)": round(record["mean_query_ms"], 3),
        "Neighbor preparation (s)": round(
            record["neighbor_preparation_seconds"], 2
        )
    })

qe_efficiency = (
    pd.DataFrame(qe_rows)
    .sort_values(["Collection", "Model"])
    .reset_index(drop=True)
)

print("Query Expansion Efficiency")
display(qe_efficiency)


# 2. Linear fusion timing.
linear_rows = []

for record in hybrid_data["pairs"]:
    linear_rows.append({
        "Collection": record["collection"],
        "Model": record["model"].upper(),
        "Linear fusion (ms)": round(
            record["mean_linear_query_ms"], 3
        )
    })

linear_df = pd.DataFrame(linear_rows)


# 3. RRF timing from the original hybrid summary.
rrf_rows = []

for record in hybrid_full["results"]:
    rrf_configs = [
        config
        for config in record["configurations"]
        if config.get("method") == "rrf"
    ]

    assert len(rrf_configs) == 1, (
        f"Expected one RRF record for {record['collection']} "
        f"{record['model']}"
    )

    rrf_rows.append({
        "Collection": record["collection"],
        "Model": record["model"].upper(),
        "RRF (ms)": round(
            rrf_configs[0]["mean_query_ms"], 3
        )
    })

rrf_df = pd.DataFrame(rrf_rows)

# Combine the two hybrid timing methods.
hybrid_efficiency = (
    linear_df.merge(
        rrf_df,
        on=["Collection", "Model"],
        how="inner",
        validate="one_to_one"
    )
    .sort_values(["Collection", "Model"])
    .reset_index(drop=True)
)

print("\nHybrid Retrieval Efficiency")
display(hybrid_efficiency)


# 4. Verify coverage.
expected_pairs = {
    (collection, model)
    for collection in ["C1", "C2"]
    for model in ["M1", "M2", "GN"]
}

for table in [qe_efficiency, hybrid_efficiency]:
    actual_pairs = set(
        zip(table["Collection"], table["Model"])
    )
    assert actual_pairs == expected_pairs
    assert len(table) == 6

print("\nAll QE, linear fusion, and RRF records verified.")

print("\nTiming limitations:")
print("QE excludes one-time neighbor preparation.")
print("Hybrid excludes initial BM25 and dense retrieval searches.")
print("Hybrid timings are not end-to-end query latency.")


Query Expansion Efficiency


,Collection,Model,QE query (ms),Neighbor preparation (s)
0,C1,GN,8.243,14.84
1,C1,M1,7.561,8.68
2,C1,M2,10.593,8.23
3,C2,GN,10.828,16.72
4,C2,M1,10.829,6.95
5,C2,M2,11.384,7.82



Hybrid Retrieval Efficiency


,Collection,Model,Linear fusion (ms),RRF (ms)
0,C1,GN,2.240,2.223
1,C1,M1,2.457,2.439
2,C1,M2,2.298,2.279
3,C2,GN,1.858,1.842
4,C2,M1,1.986,1.967
5,C2,M2,1.774,1.758



All QE, linear fusion, and RRF records verified.

Timing limitations:
QE excludes one-time neighbor preparation.
Hybrid excludes initial BM25 and dense retrieval searches.
Hybrid timings are not end-to-end query latency.


#### 3.5 Key Findings

- **BM25:** Indexing required 68.44 seconds for C1 and 71.88 seconds for C2. Average query latency was 8.14 ms and 11.68 ms, respectively.
- **Dense retrieval:** Document embedding construction took approximately 69–80 seconds. Dense matrices occupied 198.67 MiB for C1 and 241.09 MiB for C2. Average query latency ranged from 3.96–6.46 ms across models and query sets.
- **Query expansion:** Average query latency ranged from 7.56–11.38 ms across both collections. One-time neighbor preparation was measured separately and excluded from query latency.
- **Hybrid retrieval:** Linear fusion and RRF showed similar processing times, approximately 1.76–2.46 ms per query. These measurements include candidate scoring and fusion or ranking but exclude the initial BM25 and dense retrieval stages.

**Timing limitation:** QE latency excludes initial neighbor preparation, while hybrid latency excludes first-stage retrieval. Therefore, the reported hybrid measurements are not end-to-end query latency and should not be directly compared with complete retrieval timings.